# FactoryMind AI — RUL v1 Model Finalization

This notebook freezes the development specification for the NASA C-MAPSS FD001 RUL module. It trains the selected estimator on all 100 training units and defines trajectory-level inference semantics for a future guarded backend prototype.

It does **not** tune the model, compare alternatives, access official test labels, modify production source, or serialize an artifact.

## 1. Recreate the Frozen Methodology

The v1 target is retrospective RUL capped at 125 cycles. Temporal features remain grouped by unit and backward-looking. The cap is a provisional modeling assumption, not a physical guarantee.

In [1]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import sklearn
import joblib
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_DIR = Path("../data/raw")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/raw")
TRAIN_PATH = DATA_DIR / "train_FD001.txt"
assert TRAIN_PATH.exists()

canonical_columns = (["unit_id", "cycle"]
                     + [f"operational_setting_{i}" for i in range(1, 4)]
                     + [f"sensor_{i}" for i in range(1, 22)])
train = pd.read_csv(TRAIN_PATH, sep=r"\s+", header=None, names=canonical_columns)
RUL_CAP = 125
train["raw_rul"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
train["capped_rul"] = train["raw_rul"].clip(upper=RUL_CAP)

constant_sensors = ["sensor_1", "sensor_5", "sensor_10", "sensor_16", "sensor_18", "sensor_19"]
near_constant_excluded = ["sensor_6"]
excluded_sensors = constant_sensors + near_constant_excluded
retained_sensors = [f"sensor_{i}" for i in range(1, 22) if f"sensor_{i}" not in excluded_sensors]
temporal_base_sensors = ["sensor_4", "sensor_7", "sensor_11", "sensor_12", "sensor_15", "sensor_21"]
raw_predictors = ["cycle", "operational_setting_1", "operational_setting_2"] + retained_sensors

def add_frozen_temporal_features(frame, unit_column="unit_id"):
    result = frame.sort_values([unit_column, "cycle"]).copy()
    grouped = result.groupby(unit_column, sort=False)
    engineered = []
    for sensor in temporal_base_sensors:
        lag_1 = f"{sensor}_lag_1"
        lag_5 = f"{sensor}_lag_5"
        rolling_mean = f"{sensor}_rolling_mean_5"
        rolling_std = f"{sensor}_rolling_std_5"
        delta_1 = f"{sensor}_delta_1"
        result[lag_1] = grouped[sensor].shift(1)
        result[lag_5] = grouped[sensor].shift(5)
        result[rolling_mean] = grouped[sensor].transform(lambda x: x.rolling(5, min_periods=1).mean())
        result[rolling_std] = grouped[sensor].transform(lambda x: x.rolling(5, min_periods=1).std(ddof=0))
        result[delta_1] = result[sensor] - result[lag_1]
        engineered.extend([lag_1, lag_5, rolling_mean, rolling_std, delta_1])
    return result, engineered

train_features, engineered_predictors = add_frozen_temporal_features(train)
proposed_predictors = raw_predictors + engineered_predictors

assert train.shape[0] == 20_631 and train["unit_id"].nunique() == 100
assert len(raw_predictors) == 17 and len(engineered_predictors) == 30
assert len(proposed_predictors) == 47 and len(set(proposed_predictors)) == 47
assert not {"unit_id", "raw_rul", "capped_rul"}.intersection(proposed_predictors)
assert not set(excluded_sensors).intersection(proposed_predictors)
assert (train["capped_rul"] <= RUL_CAP).all()
print(f"Frozen contract verified: {len(raw_predictors)} raw + {len(engineered_predictors)} temporal = {len(proposed_predictors)} predictors.")

Frozen contract verified: 17 raw + 30 temporal = 47 predictors.


## 2. Train the Final Development Estimator

This is the final estimator for the frozen v1 **development specification**, not an independently validated industrial-production model. Median imputation and missing indicators remain inside the pipeline so inference behavior matches notebooks 08–09.

In [2]:
RF_HYPERPARAMETERS = {
    "n_estimators": 200,
    "max_features": 0.7,
    "min_samples_leaf": 3,
    "random_state": 42,
    "n_jobs": -1,
}
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ]), proposed_predictors)
], remainder="drop")

final_development_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(**RF_HYPERPARAMETERS)),
])
final_development_model.fit(train_features[proposed_predictors], train_features["capped_rul"])

fitted_rf = final_development_model.named_steps["model"]
for parameter, expected in RF_HYPERPARAMETERS.items():
    assert fitted_rf.get_params()[parameter] == expected
print("Final development estimator fitted on all 20,631 observations from 100 training units.")
display(pd.Series(RF_HYPERPARAMETERS).to_frame("frozen_value"))

Final development estimator fitted on all 20,631 observations from 100 training units.


,frozen_value
n_estimators,200.0000
max_features,0.7000
min_samples_leaf,3.0000
random_state,42.0000
n_jobs,-1.0000


## 3. Inference Granularity and Raw Input Contract

The future API must accept an **engine trajectory**, not a single isolated feature row. Lag 1, lag 5, rolling statistics, and delta require history. A one-cycle trajectory can be scored technically because the trained median imputer handles undefined lags, but it contains far less degradation evidence.

The reference helper requires strictly increasing, consecutive, unique cycles. It does not silently reorder records or fabricate missing history. `unit_id` is optional because one request already represents one trajectory; when present, it must contain exactly one value.

In [3]:
required_trajectory_columns = (
    ["cycle", "operational_setting_1", "operational_setting_2", "operational_setting_3"]
    + retained_sensors
)
forbidden_inference_columns = {"raw_rul", "capped_rul", "RUL", "official_RUL", "target"}

def predict_latest_rul(model, trajectory_df):
    if not isinstance(trajectory_df, pd.DataFrame) or trajectory_df.empty:
        raise ValueError("trajectory_df must be a non-empty pandas DataFrame.")
    missing = sorted(set(required_trajectory_columns) - set(trajectory_df.columns))
    if missing:
        raise ValueError(f"Missing required trajectory columns: {missing}")
    forbidden = sorted(forbidden_inference_columns.intersection(trajectory_df.columns))
    if forbidden:
        raise ValueError(f"Target fields are not accepted for inference: {forbidden}")
    if "unit_id" in trajectory_df and trajectory_df["unit_id"].nunique(dropna=False) != 1:
        raise ValueError("A trajectory request must contain exactly one unit_id.")

    working = trajectory_df.copy()
    numeric_columns = required_trajectory_columns
    if working[numeric_columns].isna().any().any():
        raise ValueError("Raw trajectory inputs cannot contain missing values.")
    if not np.isfinite(working[numeric_columns].to_numpy(dtype=float)).all():
        raise ValueError("Raw trajectory inputs must be finite numeric values.")
    if working["cycle"].duplicated().any():
        raise ValueError("Duplicate cycles are not allowed.")
    if not working["cycle"].is_monotonic_increasing:
        raise ValueError("Cycles must be supplied in strictly increasing order.")
    if (working["cycle"].diff().dropna() != 1).any():
        raise ValueError("Cycles must be consecutive for the frozen lag semantics.")
    if (working["cycle"] <= 0).any():
        raise ValueError("Cycle values must be positive.")

    internal_unit = working["unit_id"] if "unit_id" in working else pd.Series(1, index=working.index)
    working = working.assign(unit_id=internal_unit.to_numpy())
    engineered, generated_names = add_frozen_temporal_features(working)
    assert generated_names == engineered_predictors
    latest_features = engineered.iloc[[-1]][proposed_predictors]
    raw_prediction = float(model.predict(latest_features)[0])
    if not np.isfinite(raw_prediction):
        raise RuntimeError("Model returned a non-finite RUL estimate.")
    return raw_prediction

input_contract = pd.DataFrame({
    "field": required_trajectory_columns,
    "role": (["cycle ordering"] + ["operating setting"] * 3 + ["retained sensor measurement"] * len(retained_sensors)),
    "required": True,
})
display(input_contract)
print("unit_id: optional grouping identity; the request itself can supply trajectory identity.")

,field,role,required
0,cycle,cycle ordering,True
1,operational_setting_1,operating setting,True
2,operational_setting_2,operating setting,True
3,operational_setting_3,operating setting,True
4,sensor_2,retained sensor measurement,True
5,sensor_3,retained sensor measurement,True
6,sensor_4,retained sensor measurement,True
7,sensor_7,retained sensor measurement,True
8,sensor_8,retained sensor measurement,True
9,sensor_9,retained sensor measurement,True


unit_id: optional grouping identity; the request itself can supply trajectory identity.


## 4. Short- and Long-History Validation

The same engine is truncated to controlled history lengths. Its retrospective target is not passed to the helper. These tests demonstrate technical scoreability, not accuracy at each history length.

In [4]:
reference_unit_id = 26
reference_raw = train.loc[train["unit_id"].eq(reference_unit_id), canonical_columns].copy()
history_lengths = [1, 3, 5, 20, len(reference_raw)]
history_labels = ["one cycle", "three cycles", "five cycles", "longer trajectory", "full trajectory"]
history_results = []
for label, length in zip(history_labels, history_lengths):
    trajectory = reference_raw.head(length)
    prediction = predict_latest_rul(final_development_model, trajectory)
    history_results.append({
        "history": label,
        "observed_cycles": length,
        "latest_cycle": int(trajectory["cycle"].iloc[-1]),
        "raw_model_prediction": prediction,
        "finite_scalar": np.isscalar(prediction) and np.isfinite(prediction),
        "within_capped_target_range": 0 <= prediction <= RUL_CAP,
    })
history_results = pd.DataFrame(history_results)
display(history_results)
assert history_results["finite_scalar"].all()
assert history_results["within_capped_target_range"].all()

# Behavioral validation for malformed trajectory requests.
def expect_value_error(frame):
    try:
        predict_latest_rul(final_development_model, frame)
    except ValueError:
        return True
    return False
assert expect_value_error(reference_raw.head(3).iloc[::-1])
assert expect_value_error(pd.concat([reference_raw.head(2), reference_raw.iloc[[1]]], ignore_index=True))
assert expect_value_error(pd.concat([reference_raw.head(2), train.loc[train["unit_id"].eq(27), canonical_columns].head(1)]))
assert expect_value_error(reference_raw.head(3).drop(columns=["sensor_11"]))
assert expect_value_error(reference_raw.head(3).assign(raw_rul=10))
print("Invalid ordering, duplicates, multiple units, missing inputs, and target fields are rejected.")

,history,observed_cycles,latest_cycle,raw_model_prediction,finite_scalar,within_capped_target_range
0,one cycle,1,1,124.8542,True,True
1,three cycles,3,3,124.9717,True,True
2,five cycles,5,5,124.5120,True,True
3,longer trajectory,20,20,124.8217,True,True
4,full trajectory,199,199,1.3094,True,True


Invalid ordering, duplicates, multiple units, missing inputs, and target fields are rejected.


### Minimum practical history

- **1 cycle:** technically scoreable; lag 1, lag 5, and delta are unavailable and imputed. Treat as low-context.
- **2–4 cycles:** lag 1 and deltas become available after the first row; lag 5 is still unavailable.
- **Exactly 5 cycles:** still no lag-5 value at the latest row because a five-observation lag requires a sixth observation.
- **6+ consecutive cycles:** the latest row has the complete frozen feature families. This is the recommended minimum for a full-context v1 estimate.
- **Longer histories:** only the latest five observations directly affect fixed-window features, while cycle and current readings retain current-state context.

Short histories should not be padded with invented values. The API should return a deterministic history-quality field such as `limited_history` for fewer than six cycles.

## 5. Prediction Range and Cap Presentation

A Random Forest prediction is an average of training-target values, so this fitted estimator is mathematically bounded by the observed capped training target range. Training-fit predictions below are diagnostics only—not generalization metrics.

In [5]:
training_fit_predictions = final_development_model.predict(train_features[proposed_predictors])
range_diagnostic = pd.Series({
    "training_target_min": train_features["capped_rul"].min(),
    "training_target_max": train_features["capped_rul"].max(),
    "training_fit_prediction_min": training_fit_predictions.min(),
    "training_fit_prediction_max": training_fit_predictions.max(),
    "predictions_at_or_above_120": int((training_fit_predictions >= 120).sum()),
    "percentage_at_or_above_120": (training_fit_predictions >= 120).mean() * 100,
})
display(range_diagnostic.to_frame("training_diagnostic"))
assert training_fit_predictions.min() >= 0
assert training_fit_predictions.max() <= RUL_CAP

DISPLAY_CAP_REGION_START = 120
print(f"Proposed long-horizon presentation region begins at {DISPLAY_CAP_REGION_START} predicted cycles.")

,training_diagnostic
training_target_min,0.0000
training_target_max,125.0000
training_fit_prediction_min,0.3104
training_fit_prediction_max,125.0000
predictions_at_or_above_120,"6,696.0000"
percentage_at_or_above_120,32.4560


Proposed long-horizon presentation region begins at 120 predicted cycles.


A raw prediction below 120 may be displayed as a rounded **Estimated Remaining Useful Life** point estimate. At or above 120, v1 should avoid false precision and display **“125+ cycle horizon / long remaining-life region”** while retaining the internal numeric estimate for logging.

The research helper returns the raw model prediction and does not clamp it. The future API may defensively bound presentation output to [0, 125] after validating finiteness, but must log any out-of-range raw value as an invariant violation rather than silently hiding it.

## 6. Proposed API Output Semantics

The response should describe a development-stage estimate rather than exact remaining life.

In [6]:
proposed_response_contract = pd.DataFrame([
    ("predicted_rul_cycles", "number or null", "Rounded capped-RUL point estimate; null in long-horizon display mode if desired"),
    ("rul_display", "string", "Estimated Remaining Useful Life wording; uses 125+ horizon near the cap"),
    ("prediction_horizon_cap", "integer", "125-cycle provisional modeling cap"),
    ("history_cycle_count", "integer", "Number of observations supplied"),
    ("history_quality", "string", "limited_history for <6 cycles; full_temporal_context for 6+"),
    ("model_version", "string", "Version of the frozen RUL specification"),
    ("dataset", "string", "NASA C-MAPSS FD001"),
    ("development_stage", "boolean", "Always true for RUL v1"),
    ("warning", "string", "Stable near-failure overestimation warning"),
    ("disclaimer", "string", "Point estimate is not guaranteed life or a safety certification"),
], columns=["field", "type", "meaning"])
display(proposed_response_contract)

GLOBAL_WARNING = (
    "RUL estimates may overestimate remaining life, particularly near failure. "
    "Use alongside inspection, maintenance history, and other engineering evidence."
)
DISCLAIMER = (
    "Development-stage point estimate from simulated FD001 data. No calibrated uncertainty "
    "interval or guaranteed minimum remaining life is available."
)
print("Stable warning returned with every prediction:")
print(GLOBAL_WARNING)

,field,type,meaning
0,predicted_rul_cycles,number or null,Rounded capped-RUL point estimate; null in lon...
1,rul_display,string,Estimated Remaining Useful Life wording; uses ...
2,prediction_horizon_cap,integer,125-cycle provisional modeling cap
3,history_cycle_count,integer,Number of observations supplied
4,history_quality,string,limited_history for <6 cycles; full_temporal_c...
5,model_version,string,Version of the frozen RUL specification
6,dataset,string,NASA C-MAPSS FD001
7,development_stage,boolean,Always true for RUL v1
8,warning,string,Stable near-failure overestimation warning
9,disclaimer,string,Point estimate is not guaranteed life or a saf...


Stable warning returned with every prediction:
RUL estimates may overestimate remaining life, particularly near failure. Use alongside inspection, maintenance history, and other engineering evidence.


The warning should be returned **with every prediction**, not conditionally triggered by the predicted value. Notebook 09 showed that true near-failure cases can be overestimated, so the model cannot reliably know when the warning is most needed. Stable deterministic language is safer and easier for clients to preserve.

## 7. Uncertainty Limitations

RUL v1 provides only a point estimate. It does not provide calibrated prediction intervals, confidence bounds, a guaranteed minimum life, or a probability of surviving a specified horizon.

Future research may consider quantile regression, split conformal prediction with unit-aware calibration, or carefully validated ensemble-based dispersion. None is implemented or implied here.

## 8. Future Artifact Metadata Specification

In [7]:
package_versions = {
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "joblib": joblib.__version__,
}
metadata_specification = {
    "model_name": "FactoryMind RUL FD001 Random Forest",
    "model_version": "1.0.0-development",
    "model_family": "RandomForestRegressor",
    "dataset": "NASA C-MAPSS",
    "dataset_subset": "FD001",
    "target": "capped_rul",
    "rul_cap": RUL_CAP,
    "raw_input_columns": required_trajectory_columns,
    "predictor_columns": proposed_predictors,
    "excluded_sensors": excluded_sensors,
    "temporal_base_sensors": temporal_base_sensors,
    "temporal_feature_definitions": {
        "lag_1": "positive group shift of 1 observation",
        "lag_5": "positive group shift of 5 observations",
        "rolling_mean_5": "trailing window 5, min_periods=1",
        "rolling_std_5": "trailing window 5, min_periods=1, ddof=0",
        "delta_1": "current value minus lag_1",
    },
    "random_forest_hyperparameters": RF_HYPERPARAMETERS,
    "training_unit_count": 100,
    "training_row_count": 20_631,
    "groupkfold_metrics_notebook_08": {
        "rmse_mean": 16.3588, "rmse_std": 0.8122,
        "mae_mean": 11.1010, "mae_std": 0.6563,
        "r2_mean": 0.8456, "r2_std": 0.0149,
    },
    "official_endpoint_metrics_notebook_09": {
        "capped_rmse": 17.5413, "capped_mae": 13.0300, "capped_r2": 0.8084,
        "nasa_total_score": 522.9481, "nasa_mean_penalty": 5.2295,
        "near_failure_rmse": 14.1842, "near_failure_mae": 8.8359,
        "near_failure_mean_signed_error": 7.0016,
    },
    "known_limitations": [
        "Simulated FD001 data with one operating condition and one fault mode",
        "125-cycle target cap is provisional",
        "Point estimates only; no calibrated uncertainty",
        "Observed tendency to overestimate RUL near failure",
        "Official FD001 test set is development-exposed",
        "No external real-fleet validation",
    ],
    "output_interpretation": "Development-stage capped-RUL point estimate from the latest trajectory observation",
    "development_stage_warning": GLOBAL_WARNING,
    "package_versions": package_versions,
}

metadata_fields = pd.DataFrame({
    "field": list(metadata_specification.keys()),
    "planned_value_type": [type(value).__name__ for value in metadata_specification.values()],
})
display(metadata_fields)
display(pd.Series(package_versions).to_frame("version"))

,field,planned_value_type
0,model_name,str
1,model_version,str
2,model_family,str
3,dataset,str
4,dataset_subset,str
5,target,str
6,rul_cap,int
7,raw_input_columns,list
8,predictor_columns,list
9,excluded_sensors,list


,version
python,3.13.2
scikit_learn,1.9.0
numpy,2.5.2
pandas,3.0.5
joblib,1.5.3


Serialized scikit-learn/joblib objects are version-sensitive. The future training command should record exact package versions and the consuming runtime should validate compatible versions before loading. Determinism is supported by `random_state=42`; reproducibility also depends on the exact data, feature code, package versions, and platform-level numerical behavior.

## 9. Frozen RUL v1 Specification

| Component | Frozen v1 decision |
|---|---|
| Target | Capped RUL, cap = 125 cycles |
| Model | RandomForestRegressor: 200 trees, max_features=0.7, min_samples_leaf=3, random_state=42, n_jobs=-1 |
| Predictors | 47 total |
| Raw predictors | 17 |
| Temporal predictors | 30 |
| Temporal history | Grouped by trajectory; backward-looking only |
| Training data | FD001 training partition; 20,631 rows and 100 engines |
| Inference | Latest observation from one supplied consecutive trajectory |
| Full-context minimum | 6 cycles; shorter histories technically scoreable and flagged |
| Output | Development-stage capped-RUL point estimate |
| Cap display | Predictions ≥120 shown as “125+ cycle horizon / long remaining-life region” |
| Uncertainty | Not available in v1 |
| External validation | Not available |
| Official FD001 test | Development-exposed after notebook 09 |

No target fields enter inference. No artificial history is generated.

## 10. Production-Refactor Readiness

### Verdict A — Ready for guarded backend prototyping

The methodology is sufficiently defined and reproducible to refactor into tested reusable source code. Evidence includes stable unit-held-out validation, acceptable official endpoint performance for a development prototype, a frozen feature/model contract, and successful trajectory inference across short and long histories.

This verdict does **not** mean industrial-production-ready. Near-failure overestimation, lack of uncertainty, simulated single-condition data, and absence of external validation require persistent warnings and prevent autonomous maintenance decisions.

## 11. Validation

In [8]:
assert len(proposed_predictors) == 47
assert len(raw_predictors) == 17 and len(engineered_predictors) == 30
assert fitted_rf.get_params()["n_estimators"] == 200
assert fitted_rf.get_params()["max_features"] == 0.7
assert fitted_rf.get_params()["min_samples_leaf"] == 3
assert fitted_rf.get_params()["random_state"] == 42
assert fitted_rf.get_params()["n_jobs"] == -1
assert history_results["finite_scalar"].all()
assert history_results["within_capped_target_range"].all()
assert not forbidden_inference_columns.intersection(required_trajectory_columns)

hyperparameter_tuning_performed = False
alternative_models_evaluated = False
official_test_labels_used_for_fitting = False
artifact_serialized = False
production_source_modified = False
assert not any([
    hyperparameter_tuning_performed, alternative_models_evaluated,
    official_test_labels_used_for_fitting, artifact_serialized,
    production_source_modified,
])

validation = pd.Series({
    "47_feature_contract_reproduced": len(proposed_predictors) == 47,
    "frozen_RF_configuration_matches": all(fitted_rf.get_params()[k] == v for k, v in RF_HYPERPARAMETERS.items()),
    "short_and_long_trajectory_inference_finite": bool(history_results["finite_scalar"].all()),
    "targets_not_required_for_inference": not forbidden_inference_columns.intersection(required_trajectory_columns),
    "hyperparameter_tuning_performed": hyperparameter_tuning_performed,
    "alternative_models_evaluated": alternative_models_evaluated,
    "official_test_labels_affected_fit": official_test_labels_used_for_fitting,
    "artifact_serialized": artifact_serialized,
    "production_source_modified": production_source_modified,
})
display(validation.to_frame("result"))

,result
47_feature_contract_reproduced,True
frozen_RF_configuration_matches,True
short_and_long_trajectory_inference_finite,True
targets_not_required_for_inference,True
hyperparameter_tuning_performed,False
alternative_models_evaluated,False
official_test_labels_affected_fit,False
artifact_serialized,False
production_source_modified,False


## RUL Model Finalization Conclusions

- The exact 47-feature contract and frozen Random Forest configuration were reproduced and fitted on all 100 FD001 training engines.
- Trajectories with 1–5 cycles are technically scoreable through trained median imputation but provide incomplete temporal context. Six consecutive cycles are the recommended minimum for all feature families to be available at the latest observation.
- The reference helper requires chronological, consecutive, duplicate-free raw observations for exactly one trajectory and rejects target fields.
- Raw model output remains a capped-target point estimate. Values at or above 120 should be displayed as **“125+ cycle horizon / long remaining-life region”**, not exact 125-cycle life.
- Every response should carry the stable warning that estimates may overestimate remaining life, particularly near failure.
- No confidence interval, guaranteed minimum life, or safety certification exists in v1.
- Required future metadata includes the complete input/feature contracts, temporal definitions, frozen metrics, limitations, warnings, and package versions.
- **Verdict A:** ready to refactor for guarded backend prototyping, but not industrial-production use.

**Recommended next implementation step:** build and test `src/rul_features.py`, `src/rul_pipeline.py`, and `src/rul_train.py`, preserving this notebook's exact contracts. Do not create or expose an API endpoint until the reusable pipeline, artifact metadata, and trajectory-validation tests are complete.